# Antibody Interfaces over PDBs in Arboviruses

#### Using SYNCEDSTRUCTURE
(**A Python Class to handle, crossreference and annotate Structural Proteomics Data**)

Credits: Senci
LICENCES: all rights reserved

In [1]:
### <DEPENDENCIES and SyncedStructure Methods display> 
import re, urllib.request, requests, io, copy, os, pickle
from typing import List, Optional, Dict, Any
from pyprojroot import here
from tqdm import tqdm

import pandas as pd
import seaborn as sns
import math, string
from functools import partial

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Patch, Rectangle
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from synced_structure import SyncedStructure
from viz3d import side_by_side
from exporter import export_panels_pdf

pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)

synced_structure_options = dir(SyncedStructure)
d = [x for x in synced_structure_options if x.startswith("__")]
h = [x for x in synced_structure_options
    if x.startswith("_") and not x.startswith("__")]
e = [x for x in synced_structure_options if not x.startswith("_")]
for g in (d, h, e):
  if g:
    [print(g[i : i + 4]) for i in range(0, len(g), 4)]
    print()
### <\DEPENDENCIES and SyncedStructure Methods display>

### <HELPERS>
def get_uniprot(uid):
    u = str(uid).strip().upper()
    if not re.match(r"^([OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2})(-\d+)?$", u):
        raise ValueError()
    buf = io.StringIO(urllib.request.urlopen(f"https://rest.uniprot.org/uniprotkb/{u}.fasta").read().decode())
    return u, "".join(buf.read().splitlines()[1:])

def search_rcsb(
    organisms: Optional[List[str]] = None,
    keywords: Optional[List[str]] = None,
    title_contains: Optional[List[str]] = None,
    keywords_field: Optional[List[str]] = None,
    experimental_method: Optional[List[str]] = None,
    resolution_max: Optional[float] = None,
    resolution_min: Optional[float] = None,
    release_date_from: Optional[str] = None,
    release_date_to: Optional[str] = None,
    polymer_entity_count_min: Optional[int] = None,
    polymer_entity_count_max: Optional[int] = None,
    uniprot_ids: Optional[List[str]] = None,
    enzyme_classification: Optional[List[str]] = None,
    return_type: str = "entry",
    max_rows: int = 10000,
    return_all_hits: bool = False,
    include_computational: bool = False,
    sort_by: str = "score",
    sort_direction: str = "desc",
    timeout: int = 30,
) -> List[str]:
    """Search RCSB PDB using the Search API v2.

    Args:
        organisms: Scientific names matched with contains_phrase on
            rcsb_entity_source_organism.ncbi_scientific_name (OR).
        keywords: Free-text terms searched via full_text service (OR).
        title_contains: Words that must appear in struct.title (OR).
        keywords_field: Words that must appear in struct_keywords.pdbx_keywords (OR).
        experimental_method: Exact matches on exptl.method (OR).
        resolution_max: Upper resolution limit in Å.
        resolution_min: Lower resolution limit in Å.
        release_date_from: Earliest release date (YYYY-MM-DD).
        release_date_to: Latest release date (YYYY-MM-DD).
        polymer_entity_count_min: Minimum number of polymer entities.
        polymer_entity_count_max: Maximum number of polymer entities.
        uniprot_ids: UniProt accessions (OR).
        enzyme_classification: EC numbers (OR).
        return_type: One of entry, assembly, polymer_entity, non_polymer_entity,
            polymer_instance, mol_definition.
        max_rows: Maximum results to return (ignored if return_all_hits=True).
        return_all_hits: Return every matching identifier.
        include_computational: Also include Computed Structure Models.
        sort_by: Sort field (score, size, code, date, ...).
        sort_direction: asc or desc.
        timeout: Request timeout in seconds.

    Returns:
        List of matching identifiers.

    Raises:
        ValueError: If no search criteria are provided.
        requests.HTTPError: On non-2xx API response.
    """
    def t(attr, op, val, neg=False):
        return {"type": "terminal", "service": "text", "parameters": {"attribute": attr, "operator": op, "negation": neg, "value": val}}
    def ft(val):
        return {"type": "terminal", "service": "full_text", "parameters": {"value": val}}
    def g(nodes, op="or"):
        return nodes[0] if len(nodes) == 1 else {"type": "group", "logical_operator": op, "nodes": nodes}

    nodes = []
    if organisms:
        nodes.append(g([t("rcsb_entity_source_organism.ncbi_scientific_name", "contains_phrase", o) for o in organisms]))
    if keywords:
        nodes.append(g([ft(k) for k in keywords]))
    if title_contains:
        nodes.append(g([t("struct.title", "contains_words", x) for x in title_contains]))
    if keywords_field:
        nodes.append(g([t("struct_keywords.pdbx_keywords", "contains_words", x) for x in keywords_field]))
    if experimental_method:
        nodes.append(g([t("exptl.method", "exact_match", m) for m in experimental_method]))
    if resolution_min is not None or resolution_max is not None:
        nodes.append(t("rcsb_entry_info.resolution_combined", "range", {
            "from": resolution_min if resolution_min is not None else 0.0,
            "to": resolution_max if resolution_max is not None else 100.0,
            "include_lower": True, "include_upper": True}))
    if release_date_from or release_date_to:
        dv = {"include_lower": True, "include_upper": True}
        if release_date_from: dv["from"] = release_date_from
        if release_date_to: dv["to"] = release_date_to
        nodes.append(t("rcsb_accession_info.initial_release_date", "range", dv))
    if polymer_entity_count_min is not None or polymer_entity_count_max is not None:
        nodes.append(t("rcsb_entry_info.polymer_entity_count", "range", {
            "from": polymer_entity_count_min if polymer_entity_count_min is not None else 0,
            "to": polymer_entity_count_max if polymer_entity_count_max is not None else 9999,
            "include_lower": True, "include_upper": True}))
    if uniprot_ids:
        nodes.append(g([t("rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers.database_accession", "exact_match", u) for u in uniprot_ids]))
    if enzyme_classification:
        nodes.append(g([t("rcsb_polymer_entity.rcsb_enzyme_class_combined.ec", "exact_match", e) for e in enzyme_classification]))
    if not nodes:
        raise ValueError("At least one search criterion must be provided")

    query = g(nodes, "and")
    content = ["experimental"] + (["computational"] if include_computational else [])
    opts: Dict[str, Any] = {"results_content_type": content, "sort": [{"sort_by": sort_by, "direction": sort_direction}], "scoring_strategy": "combined"}
    if return_all_hits:
        opts["return_all_hits"] = True
    else:
        opts["paginate"] = {"start": 0, "rows": max_rows}

    r = requests.post("https://search.rcsb.org/rcsbsearch/v2/query", json={"query": query, "return_type": return_type, "request_options": opts}, timeout=timeout)
    r.raise_for_status()
    return [h["identifier"] for h in r.json().get("result_set", [])]

def get_pdb_metadata(pdb_id) -> pd.DataFrame:
    ids = pdb_id.tolist() if isinstance(pdb_id, pd.Series) else (pdb_id if isinstance(pdb_id, list) else [pdb_id])
    rows = []
    for pid in tqdm(ids):
        p = str(pid).strip().upper()
        row = {"pdb_id": p, "title": None, "journal": None, "year": None, "doi": None, "pubmed_id": None, "authors": None, "citation_title": None}
        try:
            r = requests.get(f"https://data.rcsb.org/rest/v1/core/entry/{p}", timeout=30)
            r.raise_for_status()
            d = r.json()
            row["title"] = d.get("struct", {}).get("title")
            citations = d.get("citation", [])
            primary = next((c for c in citations if c.get("rcsb_is_primary") == "Y"), citations[0] if citations else {})
            if primary:
                row["journal"] = primary.get("journal_abbrev")
                row["year"] = primary.get("year")
                row["doi"] = primary.get("pdbx_database_id_DOI")
                row["pubmed_id"] = primary.get("pdbx_database_id_PubMed")
                row["authors"] = primary.get("rcsb_authors")
                row["citation_title"] = primary.get("title")
        except Exception:
            pass
        rows.append(row)
    return pd.DataFrame(rows)
    
### <\HELPERS>

['__class__', '__contains__', '__delattr__', '__dict__']
['__dir__', '__doc__', '__eq__', '__firstlineno__']
['__format__', '__ge__', '__getattribute__', '__getitem__']
['__getstate__', '__gt__', '__hash__', '__init__']
['__init_subclass__', '__iter__', '__le__', '__len__']
['__lt__', '__module__', '__ne__', '__new__']
['__reduce__', '__reduce_ex__', '__repr__', '__setattr__']
['__sizeof__', '__static_attributes__', '__str__', '__subclasshook__']
['__weakref__']

['_angles_overlay', '_apply_chain_options', '_atom_contacts', '_atom_pair_maps']
['_auth_of', '_block_entries', '_build_chains', '_ca_records']
['_chain_colors', '_chain_description_map', '_chain_entity_map', '_chain_geometry']
['_chain_name', '_checked_matrix', '_clean_val', '_compute_interface']
['_compute_local_alignment', '_coords_touch', '_draw_pair', '_entity_sequences']
['_export', '_external_refs_for_entity', '_gray_matrix', '_heavy_atoms']
['_hydrophobicity', '_keep_mapped', '_label_of', '_local_alignment']
['_make_ca

# Antibody Interfaces over PDBs in Arboviruses

```
Signature:
search_rcsb(
    organisms: Optional[List[str]] = None,
    keywords: Optional[List[str]] = None,
    title_contains: Optional[List[str]] = None,
    keywords_field: Optional[List[str]] = None,
    experimental_method: Optional[List[str]] = None,
    resolution_max: Optional[float] = None,
    resolution_min: Optional[float] = None,
    release_date_from: Optional[str] = None,
    release_date_to: Optional[str] = None,
    polymer_entity_count_min: Optional[int] = None,
    polymer_entity_count_max: Optional[int] = None,
    uniprot_ids: Optional[List[str]] = None,
    enzyme_classification: Optional[List[str]] = None,
    return_type: str = 'entry',
    max_rows: int = 10000,
    return_all_hits: bool = False,
    include_computational: bool = False,
    sort_by: str = 'score',
    sort_direction: str = 'desc',
    timeout: int = 30,
) -> List[str]
Docstring:
Search RCSB PDB using the Search API v2.
```


### Search

Query to RCSB returns 66 PDB IDS. We will use UniProtKB standard entry for ZIKV Polyprotein (`A0A142I5B9`) using specific range for envelope (`291-794`)


In [9]:
full_ref = list(get_uniprot('A0A142I5B9'))
rng = [291, 794]
ns1_rng = [795, 1146]
full_seq = full_ref[1]
env_seq = full_seq[rng[0] - 1 : rng[1]]
ns1_seq = full_seq[ns1_rng[0] - 1 : ns1_rng[1]]
ref_id = full_ref[0]

z_ids = search_rcsb(organisms=["Zika virus"])
z_abs = search_rcsb(organisms=["Zika virus"], keywords=["antibody", "Fab", "scFv", "mAb", "nanobody"])
z_ids = [x for x in z_ids if x != "7T17"]
z_abs = [x for x in z_abs if x != "7T17"]


def ab_match(t):
    return bool(re.search('Fab|heavy|light|scfv|VL|VH|Ig', str(t), re.I))

print(f"ZIKA {len(z_ids)}")
print(f"ZIKA & Abs {len(z_abs)}")

out_dir = here() / 'data/ZIKA-INTERFACES'
out_dir.mkdir(parents=True, exist_ok=True)
print(f"{ref_id}: {full_seq[:60]}...")
print(z_abs)

OUT = out_dir / 'interfaces'
cov_cut = 2.5

data = {}
summaries = []
count = 0

for pdb in z_abs:
    p_dir = OUT / pdb
    p_dir.mkdir(parents=True, exist_ok=True)
    pkl = p_dir / f"{pdb}_sync.pkl"
    
    if pkl.exists():
        with open(pkl, 'rb') as f:
            sync = pickle.load(f)
    else:
        sync = SyncedStructure.from_rcsb_id(pdb)
        biotypes = sync.moiety_summary
        non_prot = biotypes[biotypes['moiety'] != 'protein']['auth_chain_id'].unique()
        for np in non_prot:
            sync.remove_chain(np)
            
        e_df = sync.match_chains(full_seq, min_identity=5)
        e_chains = e_df[e_df['query_coverage_percent'] > cov_cut]['auth_chain_id']
        for z in e_chains:
            sync.set_external_ref(chain_id=z, external_ref=ref_id)
            sync.remap_chain(z)
            
        f_df = sync.match_chains(env_seq, min_identity=5)
        f_chains = f_df[f_df['query_coverage_percent'] > cov_cut]['auth_chain_id']
        for z in f_chains:
            sync.set_reference_sequence(chain_id=z, sequence=env_seq, remap=True)
            sync.set_external_ref(chain_id=z, external_ref=f"ENV_{ref_id}")
            sync.remap_chain(z)

        n_df = sync.match_chains(ns1_seq, min_identity=5)
        n_chains = n_df[n_df['query_coverage_percent'] > cov_cut]['auth_chain_id']
        for z in n_chains:
            sync.set_reference_sequence(chain_id=z, sequence=ns1_seq, remap=True)
            sync.set_external_ref(chain_id=z, external_ref=f"NS1_{ref_id}")
            sync.remap_chain(z)
            
        with open(pkl, 'wb') as f:
            pickle.dump(sync, f)
            
    summary = sync.summary()
    summary['pdb_id'] = pdb
    e_df = sync.match_chains(full_seq, min_identity=5)
    e_chains = e_df[e_df['query_coverage_percent'] > cov_cut]['auth_chain_id']
    f_df = sync.match_chains(env_seq, min_identity=5)
    f_chains = f_df[f_df['query_coverage_percent'] > cov_cut]['auth_chain_id']
    n_df = sync.match_chains(ns1_seq, min_identity=5)
    n_chains = n_df[n_df['query_coverage_percent'] > cov_cut]['auth_chain_id']
    summary['is_polyprotein'] = summary['auth_chain_id'].isin(e_chains)
    summary['is_env'] = summary['auth_chain_id'].isin(f_chains)
    summary['is_ns1'] = summary['auth_chain_id'].isin(n_chains)
    summary['is_ab'] = summary['description'].apply(ab_match)
    data[pdb] = sync
    summaries.append(summary)
    
    count += 1
    if count % 10 == 0:
        print(f"[PYTHON-INFO] {count} done with PDBs processed")

ZIKA 421
ZIKA & Abs 65
A0A142I5B9: MKNPKKKSGGFRIVNMLKRGVARVSPFGGLKRLPAGLLLGHGPIRMVLAILAFLRFTAIK...
['7A3N', '8DV6', '5UHY', '5KVG', '5KVF', '8WNP', '8WNU', '8WO4', '5KVD', '7A3U', '5KVE', '8JQN', '8JQM', '7KCR', '6JEP', '7BSD', '8JKH', '5Y0A', '7CBP', '5VIG', '6DFI', '5GZR', '8CXG', '8JQS', '7BUA', '9XYB', '8JKF', '6UTA', '7BU8', '7C2T', '9PM6', '5LBV', '5JHL', '6NIU', '5GZN', '9OD2', '5OMZ', '5LBS', '5JHM', '8WO0', '5H32', '5LCV', '6PLK', '8EE5', '7BPK', '5GZO', '5H30', '9VWM', '8WN8', '9OWF', '6JFH', '6MID', '6LNT', '8CXH', '8CXI', '5H37', '9JS7', '7BQ5', '6JFI', '9JS6', '8EE8', '8EED', '8EEE', '7YAR', '6NIP']
[PYTHON-INFO] 10 done with PDBs processed
[PYTHON-INFO] 20 done with PDBs processed
[PYTHON-INFO] 30 done with PDBs processed
[PYTHON-INFO] 40 done with PDBs processed
[PYTHON-INFO] 50 done with PDBs processed
[PYTHON-INFO] 60 done with PDBs processed


In [10]:
tsv_path = OUT / 'zikv_model_summaries.tsv'

if tsv_path.exists():
    merged_df = pd.read_csv(tsv_path, sep="\t")
    sync_df = merged_df
else:
    sync_df = pd.concat(summaries, ignore_index=True)
    sync_metadata = get_pdb_metadata(pdb_id=list(set(sync_df['pdb_id'])))
    sync_df = pd.merge(sync_df, sync_metadata, on='pdb_id', how='left')
    sync_df.to_csv(tsv_path, sep="\t", index=False)

In [11]:
###############################################################################

### Interface Retrieval

**Interface Definition hereby**
*possessed by a biomolymeric complex*. At least a couple of chains that are in contact.

**Step 1 - Reconstruction of Chain Partnership**

**Step 2 - Interaction type**


In [12]:
import pandas as pd

records = []
seen = set()

for p in partners_dict:
    # Use .copy() to prevent any in-place DataFrame modification issues across multiple runs
    p_df = sync_df[sync_df['pdb_id'] == p].copy()
    
    # 1. Filtra le catene Envelope con external_ref che inizia per 'ENV_'
    env_chains = p_df[p_df['external_ref'].fillna('').str.startswith('ENV_')]['chain_id'].tolist()
    if not env_chains:
        continue
        
    # 2. Costruisci i duos di anticorpi
    ab_df = p_df[p_df['is_ab'] | p_df['description'].apply(ab_match)]
    h_chains = ab_df[ab_df['description'].str.contains('heavy|VH|fab heavy|chain 1', case=False, na=False)]['chain_id'].tolist()
    l_chains = ab_df[ab_df['description'].str.contains('light|VL|fab light|chain 2', case=False, na=False)]['chain_id'].tolist()
    others = ab_df[~ab_df['chain_id'].isin(h_chains + l_chains)]['chain_id'].tolist()
    
    paired_l, paired_h, duos = set(), set(), []
    for h in h_chains:
        match_l = [l for l in l_chains if l not in paired_l and l in partners_dict.get(p, {}).get(h, set())]
        if match_l:
            l = match_l[0]
            paired_l.add(l)
            paired_h.add(h)
            duos.append(([h], [l]))
            
    for h in [x for x in h_chains if x not in paired_h]:
        for l in [y for y in l_chains if y not in paired_l]:
            duos.append(([h], [l]))
            paired_h.add(h)
            paired_l.add(l)
            break
            
    for h in [x for x in h_chains if x not in paired_h]:
        duos.append(([h], []))
    for l in [x for x in l_chains if x not in paired_l]:
        duos.append(([], [l]))
    for o in others:
        duos.append(([o], []))

    # 3. Associa i duos all'Envelope se almeno una catena è tra i partner
    for idx, (h_l, l_l) in enumerate(duos):
        duo_c = set(h_l + l_l)
        if not duo_c:
            continue
        ent_id = f"{p}_IgDuo_{idx}"
        
        for env_c in env_chains:
            env_parts = partners_dict.get(p, {}).get(env_c, set())
            is_in_contact = any(dc in env_parts or env_c in partners_dict.get(p, {}).get(dc, set()) for dc in duo_c)
            
            if is_in_contact:
                v_info = chain_info.get((p, env_c), {})
                key = (p, env_c, ent_id)
                if key not in seen:
                    seen.add(key)
                    records.append({
                        'pdb_id': p,
                        'env_chain': env_c,
                        'env_desc': v_info.get('description', 'Unknown'),
                        'heavy_chain': ", ".join(sorted(h_l)) or "None",
                        'light_chain': ", ".join(sorted(l_l)) or "None",
                        'ab_chains': ", ".join(sorted(duo_c)),
                        'ab_descriptions': " / ".join(sorted(list({chain_info.get((p, c), {}).get('description', 'Unknown') for c in duo_c})))
                    })

df_env_duos = pd.DataFrame(records).drop_duplicates().reset_index(drop=True)

print(f"Totale interfacce Envelope - Ig Duo: {len(df_env_duos)}")
print(df_env_duos.head(15))

NameError: name 'partners_dict' is not defined

### Interface Analysis
Up to now:
- We queried and retrieved 66 antibody-ZIKV antibody entries on PDB
- We remapped all protein chains matching to `A0A142I5B9` and flagged envelopes using specific range for envelope (`291-794`).
- We have deduced the antibody dimers and their interaction with Envelope and NS1 proteins\
\
Based on that:
- 100 Envelope - antibody complexes over 30 unique PDB ids
- 16 Non-structural Protein 1 - antibody complexes over 6 unique PBD ids
 
Now we can proceed to the analysis of the contact maps.

In [ ]:
import pandas as pd

# Funzione per verificare il contatto bidirezionale tra due catene
def is_connected(p, c1, c2):
    parts = partners_dict.get(p, {})
    return c2 in parts.get(c1, set()) or c1 in parts.get(c2, set())

# 1. Estrai i veri dimeri Ab-Ab (escludendo H-L dello stesso Fab)
ab_dimers = []
for p, parts in partners_dict.items():
    p_df = sync_df[sync_df['pdb_id'] == p]
    ab_df = p_df[p_df['is_ab'] | p_df['description'].apply(ab_match)]
    
    h_chains = ab_df[ab_df['description'].str.contains('heavy|VH|fab heavy|chain 1', case=False, na=False)]['chain_id'].tolist()
    l_chains = ab_df[ab_df['description'].str.contains('light|VL|fab light|chain 2', case=False, na=False)]['chain_id'].tolist()
    
    same_fab = {(h, l) for h in h_chains for l in l_chains if is_connected(p, h, l)}
    ab_chains = ab_df['chain_id'].tolist()
    
    for i in range(len(ab_chains)):
        for j in range(i + 1, len(ab_chains)):
            c1, c2 = ab_chains[i], ab_chains[j]
            if is_connected(p, c1, c2):
                if (c1, c2) in same_fab or (c2, c1) in same_fab:
                    continue
                ab_dimers.append({
                    'pdb_id': p,
                    'chain_1': c1, 'desc_1': chain_info.get((p, c1), {}).get('description', 'Unknown'),
                    'chain_2': c2, 'desc_2': chain_info.get((p, c2), {}).get('description', 'Unknown')
                })

df_ab_dimers = pd.DataFrame(ab_dimers).drop_duplicates().reset_index(drop=True)

# 2. Associa i dimeri Ab alle catene Envelope (external_ref che inizia con 'ENV_')
env_trios = []
for p, parts in partners_dict.items():
    p_df = sync_df[sync_df['pdb_id'] == p]
    env_chains = p_df[p_df['external_ref'].fillna('').str.startswith('ENV_')]['chain_id'].tolist()
    if not env_chains:
        continue
        
    p_dimers = df_ab_dimers[df_ab_dimers['pdb_id'] == p]
    if p_dimers.empty:
        continue
        
    for env_c in env_chains:
        for _, d in p_dimers.iterrows():
            c1, c2 = d['chain_1'], d['chain_2']
            if is_connected(p, env_c, c1) or is_connected(p, env_c, c2):
                env_trios.append({
                    'pdb_id': p,
                    'env_chain': env_c,
                    'ab_chain_1': c1, 'ab_desc_1': d['desc_1'],
                    'ab_chain_2': c2, 'ab_desc_2': d['desc_2']
                })

df_env_trios = pd.DataFrame(env_trios).drop_duplicates().reset_index(drop=True)

print(f"Totale dimeri Ab: {len(df_ab_dimers)}")
print(f"Totale interfacce Envelope - Dimero Ab: {len(df_env_trios)}")
print(df_env_trios.head(10))

In [ ]:
first_key, first_value = next(iter(immune_interfaces.items()))

In [ ]:
first_value.jupy_viz()